# LeetCode 3461 — Check If Digits Are Equal in String After Operations I bằng Python

Link đề: https://leetcode.com/problems/check-if-digits-are-equal-in-string-after-operations-i/ · Độ khó: **Easy** · Chủ đề: Math, String, Simulation (mô phỏng), Combinatorics (tổ hợp), Number Theory

Skeleton LeetCode:

```python
class Solution:
    def hasSameDigits(self, s: str) -> bool:
        ...
```

Ghi chú: hai ví dụ chính lấy từ đề; mọi đáp án trong notebook đã được kiểm chứng bằng code (đối chiếu năm cách làm trên hàng nghìn chuỗi chữ số ngẫu nhiên). Ví dụ nào ghi "tự dựng" là do mình thêm để minh họa — nếu cần khớp từng chữ thì nên đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho chuỗi chữ số `s`; **lặp** thao tác "thay `s` bằng dãy các **tổng hai chữ số liền kề, lấy dư cho `10`**" cho đến khi `s` còn **đúng hai chữ số**; trả về `True` nếu hai chữ số cuối cùng **giống nhau**.
2. Nắm các công cụ Python: đổi ký tự số thành số nguyên (`int(c)`, `list(map(int, s))`), **chia lấy dư** `% 10`, duyệt **cặp liền kề** (`range(len(t) - 1)`, `zip(t, t[1:])`), vòng `while len(t) > 2`, **cập nhật tại chỗ** (in-place) trong list, ghép lại chuỗi bằng `"".join(map(str, ...))`, và `math.comb`.
3. Đi từ cách **mô phỏng bằng chuỗi mới sau mỗi bước** → **mô phỏng tại chỗ trên list** → **`zip` + list comprehension** → **công thức tổ hợp** (hệ số tam giác Pascal) → **hệ số Pascal tính bằng cộng dần `mod 10`**.
4. Hiểu mấu chốt: mỗi bước làm chuỗi ngắn đi **`1`**, nên cần đúng `n - 2` bước; mỗi bước tốn `O(độ dài)` nên tổng `O(n²)` — với `n <= 100` là quá đủ. Ngoài ra còn có công thức đẹp: chữ số cuối là **tổ hợp tuyến tính** của chữ số đầu với hệ số **Pascal** (`C(m, i)`), sẽ giải thích ở mục 6–7.
5. Tránh các lỗi kinh điển: **cộng ký tự** thay vì cộng số (`"3" + "9" = "39"`), quên `% 10`, dừng sai (`len > 1` thay vì `len > 2`), **cập nhật tại chỗ theo thứ tự sai** (duyệt ngược làm dùng nhầm giá trị đã cập nhật), và chuyển cả chuỗi thành `int` làm **mất số `0` đứng đầu**.

## 2. Đề bài & ví dụ

Cho chuỗi `s` gồm các chữ số. Thực hiện thao tác sau **lặp đi lặp lại** cho đến khi chuỗi có **đúng hai** chữ số:

- Với **mỗi cặp chữ số liền kề** trong `s` (bắt đầu từ chữ số đầu tiên), tính một chữ số mới bằng **(tổng hai chữ số đó) mod 10**.
- Thay `s` bằng dãy các chữ số mới vừa tính, **giữ nguyên thứ tự** tính được.

Trả về `true` nếu hai chữ số cuối cùng của `s` **giống nhau**; ngược lại trả về `false`.

**Ví dụ 1**

- Input: `s = "3902"`
- Output: `true`
- Giải thích:
  - Ban đầu `s = "3902"`.
  - Thao tác 1: `(3 + 9) % 10 = 2`, `(9 + 0) % 10 = 9`, `(0 + 2) % 10 = 2` ⇒ `s = "292"`.
  - Thao tác 2: `(2 + 9) % 10 = 1`, `(9 + 2) % 10 = 1` ⇒ `s = "11"`.
  - Hai chữ số `"11"` giống nhau nên trả `true`.

**Ví dụ 2**

- Input: `s = "34789"`
- Output: `false`
- Giải thích: `"34789"` → `"7157"` → `"862"` → `"48"`. Vì `'4' != '8'` nên trả `false`.

**Ràng buộc:** `3 <= s.length <= 100`; `s` chỉ gồm các chữ số.

In [1]:
import random
from math import comb

examples = [
    ("3902", True),
    ("34789", False),
]
for s, expected in examples:
    print(repr(s), "-> kỳ vọng", expected)

'3902' -> kỳ vọng True
'34789' -> kỳ vọng False


## 3. Kiến thức nền

### 3.1. Từ ký tự số sang số nguyên

Mỗi phần tử của chuỗi `"3902"` là một **ký tự** (`'3'`, `'9'`, ...), **không phải số**. Dùng `int(c)` để đổi một ký tự số thành số nguyên. Muốn đổi cả chuỗi: `list(map(int, s))` (`map` áp `int` lên từng ký tự). **Cẩn thận:** `"3" + "9"` là **nối chuỗi** (`"39"`), không phải cộng số (`12`).

In [2]:
s = "3902"
print(s[0], type(s[0]))                  # 3 <class 'str'>
print(int(s[0]), type(int(s[0])))        # 3 <class 'int'>
digits = list(map(int, s))
print(digits)                            # [3, 9, 0, 2]
print("3" + "9", int("3") + int("9"))    # 39 12

3 <class 'str'>
3 <class 'int'>
[3, 9, 0, 2]
39 12


### 3.2. Phép `% 10` — lấy chữ số hàng đơn vị

`x % 10` là **phần dư** khi chia `x` cho `10`, tức **chữ số cuối** của `x` (với `x` không âm). Tổng hai chữ số lớn nhất là `9 + 9 = 18`, nên `% 10` đưa kết quả về lại `0..9`. Ví dụ: `(9 + 2) % 10 = 11 % 10 = 1`.

In [3]:
for a, b in [(3, 9), (9, 0), (0, 2), (9, 9), (5, 5)]:
    print(f"({a} + {b}) % 10 = {a + b} % 10 = {(a + b) % 10}")

(3 + 9) % 10 = 12 % 10 = 2
(9 + 0) % 10 = 9 % 10 = 9
(0 + 2) % 10 = 2 % 10 = 2
(9 + 9) % 10 = 18 % 10 = 8
(5 + 5) % 10 = 10 % 10 = 0


### 3.3. Duyệt các cặp liền kề

Với dãy `t` có `m` phần tử, các cặp liền kề là `(t[0], t[1])`, `(t[1], t[2])`, ..., `(t[m-2], t[m-1])` — tức **`m - 1` cặp**. Hai cách viết:

- chỉ số: `for i in range(len(t) - 1): t[i], t[i + 1]` (dừng ở `len(t) - 2` để `i + 1` không vượt biên);
- `zip(t, t[1:])`: ghép dãy với chính nó **dịch một bước**; `zip` dừng khi dãy ngắn hơn hết.

In [4]:
t = [3, 9, 0, 2]
print([(t[i], t[i + 1]) for i in range(len(t) - 1)])     # [(3, 9), (9, 0), (0, 2)]
print(list(zip(t, t[1:])))                                # [(3, 9), (9, 0), (0, 2)]
print(len(t) - 1, "cặp")

[(3, 9), (9, 0), (0, 2)]
[(3, 9), (9, 0), (0, 2)]
3 cặp


### 3.4. Từ dãy số về chuỗi

Để in/so sánh kết quả dưới dạng chuỗi, đổi từng số về chuỗi rồi nối: `"".join(map(str, t))`.

In [5]:
t = [2, 9, 2]
print("".join(map(str, t)))               # 292
print("".join(str(x) for x in [1, 1]))    # 11

292
11


### 3.5. Mỗi bước làm dãy ngắn đi 1 — vòng `while len(t) > 2`

Dãy `m` phần tử cho ra `m - 1` phần tử. Bắt đầu từ `n` phần tử, ta cần dừng khi còn `2`, nên số thao tác là `n - 2`. Cách viết tự nhiên: `while len(t) > 2:` (làm bước nữa khi còn **hơn hai** phần tử).

In [6]:
n = 5
t = list(range(n))
steps = 0
while len(t) > 2:
    t = t[:-1]              # mô phỏng: mỗi bước mất 1 phần tử
    steps += 1
    print("sau bước", steps, "còn", len(t), "phần tử")
print("số bước =", steps, "= n - 2 =", n - 2)

sau bước 1 còn 4 phần tử
sau bước 2 còn 3 phần tử
sau bước 3 còn 2 phần tử
số bước = 3 = n - 2 = 3


### 3.6. Cập nhật tại chỗ (in-place) và thứ tự duyệt

Thay vì tạo list mới mỗi bước, ta có thể ghi đè ngay trên `t`: `t[i] = (t[i] + t[i + 1]) % 10`. Muốn đúng, phải duyệt `i` **từ trái sang phải**: khi tính `t[i]` ta cần `t[i + 1]` **còn nguyên giá trị cũ** (nó chưa bị ghi đè). Duyệt ngược sẽ dùng nhầm giá trị đã cập nhật. Sau bước, chỉ `m - 1` phần tử đầu `t[0..m-2]` là có nghĩa (phần tử cuối bị bỏ qua).

In [7]:
t = [3, 9, 0, 2]
for i in range(len(t) - 1):               # trái -> phải
    t[i] = (t[i] + t[i + 1]) % 10
print(t[:-1])                             # [2, 9, 2]

t = [3, 9, 0, 2]
for i in range(len(t) - 2, -1, -1):       # phải -> trái: SAI
    t[i] = (t[i] + t[i + 1]) % 10
print(t[:-1], "(khác [2, 9, 2] vì dùng nhầm giá trị đã cập nhật)")

[2, 9, 2]
[4, 1, 2] (khác [2, 9, 2] vì dùng nhầm giá trị đã cập nhật)


### 3.7. Tổ hợp `math.comb` và tam giác Pascal

`math.comb(m, i)` là **số tổ hợp** `C(m, i)` ("chọn `i` trong `m`"). Các số này chính là các hàng của **tam giác Pascal**: hàng `m` là `C(m, 0), C(m, 1), ..., C(m, m)`; mỗi số bằng tổng hai số phía trên. Hàng `3`: `1, 3, 3, 1`.

In [8]:
for m in range(5):
    print(m, [comb(m, i) for i in range(m + 1)])
# 0 [1]
# 1 [1, 1]
# 2 [1, 2, 1]
# 3 [1, 3, 3, 1]
# 4 [1, 4, 6, 4, 1]

0 [1]
1 [1, 1]
2 [1, 2, 1]
3 [1, 3, 3, 1]
4 [1, 4, 6, 4, 1]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Làm đúng như đề với `s = "34789"` (`n = 5` ⇒ cần `5 - 2 = 3` thao tác):

| thao tác | chuỗi trước | các tổng `mod 10` | chuỗi sau |
|---|---|---|---|
| 1 | `34789` | `3+4=7`, `4+7=11→1`, `7+8=15→5`, `8+9=17→7` | `7157` |
| 2 | `7157` | `7+1=8`, `1+5=6`, `5+7=12→2` | `862` |
| 3 | `862` | `8+6=14→4`, `6+2=8` | `48` |

Còn `2` chữ số: `4` và `8` khác nhau ⇒ `false`.

Quan sát: mỗi lần chuỗi ngắn đi một chữ số; ta **lặp lại đúng một công thức**. Vậy cách đầu tiên chỉ là **mô phỏng trung thực** theo đề.

In [9]:
s = "34789"
level = 0
while len(s) > 2:
    nxt = ""
    for i in range(len(s) - 1):
        nxt += str((int(s[i]) + int(s[i + 1])) % 10)
    level += 1
    print(f"thao tác {level}: {s} -> {nxt}")
    s = nxt
print("hai chữ số cuối:", s, "-> giống nhau?", s[0] == s[1])

thao tác 1: 34789 -> 7157
thao tác 2: 7157 -> 862
thao tác 3: 862 -> 48
hai chữ số cuối: 48 -> giống nhau? False


## 5. Cách 1 — Brute Force / baseline: mô phỏng bằng chuỗi mới sau mỗi bước

**Ý tưởng.** Làm **đúng từng chữ của đề**:

1. Trong khi `len(s) > 2`: tạo chuỗi rỗng `new`; với mỗi `i` từ `0` đến `len(s) - 2`: tính `(int(s[i]) + int(s[i + 1])) % 10`, đổi về chuỗi và nối vào `new`.
2. Gán `s = new`.
3. Khi còn `2` chữ số: trả `s[0] == s[1]`.

In [10]:
def has_same_digits_string(s: str) -> bool:
    while len(s) > 2:
        new = ""
        for i in range(len(s) - 1):
            new += str((int(s[i]) + int(s[i + 1])) % 10)     # chữ số mới từ cặp (i, i+1)
        s = new
    return s[0] == s[1]


for s, expected in examples:
    print(has_same_digits_string(s), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )


**Độ phức tạp (Cách 1).** Gọi `n = len(s)`. Bước thứ `j` xử lý chuỗi độ dài `n - j` ⇒ tổng `(n - 1) + (n - 2) + ... + 2 ≈ n²/2` phép cộng ⇒ **Time `O(n²)`**. Mỗi bước tạo chuỗi mới ⇒ **Space `O(n)`**. Với `n <= 100` chỉ tối đa khoảng `5000` phép tính — rất nhanh.

**Nút thắt:** liên tục đổi qua lại `str ↔ int` và tạo chuỗi mới. Ta có thể đổi sang số **một lần** rồi cập nhật tại chỗ. Bên cạnh đó còn có một cách nhìn **toán học** để không cần mô phỏng cả tam giác.

## 6. Ý tưởng thuật toán / bước đột phá

Có hai nâng cấp:

**(A) Làm việc với số, cập nhật tại chỗ.** Đổi `s` thành list số `t` một lần. Ở bước còn độ dài `m`, chỉ `m - 1` phần tử đầu thay đổi: `t[i] = (t[i] + t[i + 1]) % 10` với `i = 0..m-2`. Không tạo list mới ⇒ `O(1)` bộ nhớ thêm.

**(B) Công thức tổ hợp (nhìn bài như tam giác Pascal).** Gọi `t` là dãy hiện tại. Sau `1` thao tác, mỗi phần tử mới là `t[j] + t[j+1]`. Sau `2` thao tác: `t[j] + 2·t[j+1] + t[j+2]`. Sau `3`: `t[j] + 3·t[j+1] + 3·t[j+2] + t[j+3]`. Hệ số chính là **hàng Pascal**! Tổng quát, sau `m` thao tác:

> phần tử mới thứ `j` = `Σ_{i=0..m} C(m, i) · t[j + i]` (mod 10).

Ta cần `m = n - 2` thao tác (còn đúng hai phần tử, chỉ số `j = 0` và `j = 1`):

- chữ số thứ nhất `a = Σ_{i=0..n-2} C(n-2, i) · s[i]  (mod 10)`,
- chữ số thứ hai `b = Σ_{i=0..n-2} C(n-2, i) · s[i+1]  (mod 10)`.

Ví dụ `"34789"` (`m = 3`, hàng Pascal `1, 3, 3, 1`): `a = 1·3 + 3·4 + 3·7 + 1·8 = 44 → 4`; `b = 1·4 + 3·7 + 3·8 + 1·9 = 58 → 8`. Đúng với `"48"` ở mục 4.

In [11]:
s = "34789"
t = list(map(int, s))
m = len(t) - 2
row = [comb(m, i) for i in range(m + 1)]
a = sum(row[i] * t[i] for i in range(m + 1)) % 10
b = sum(row[i] * t[i + 1] for i in range(m + 1)) % 10
print("m =", m, "| hàng Pascal:", row)
print("a =", a, "b =", b, "-> giống nhau?", a == b)         # 4 8 False

# kiểm tra "sau 2 thao tác, phần tử mới j = t[j] + 2 t[j+1] + t[j+2]"
t2 = [(x + y) % 10 for x, y in zip(t, t[1:])]
t3 = [(x + y) % 10 for x, y in zip(t2, t2[1:])]
print(t3, [(t[j] + 2 * t[j + 1] + t[j + 2]) % 10 for j in range(len(t) - 2)])

m = 3 | hàng Pascal: [1, 3, 3, 1]
a = 4 b = 8 -> giống nhau? False
[8, 6, 2] [8, 6, 2]


## 7. Các cách tiếp theo

### Cách 2 — Mô phỏng tại chỗ trên list số (cách mình khuyên nộp)

**Giải thích từng đoạn.**

1. `t = list(map(int, s))`: đổi cả chuỗi thành số **một lần**.
2. `for k in range(n - 1, 1, -1)`: `k` là **số phần tử mới** cần tính ở bước hiện tại; nó chạy `n - 1, n - 2, ..., 2` (tức `n - 2` giá trị — đúng bằng số thao tác cần làm). Bước đầu dãy có `n` phần tử nên có `n - 1` cặp; bước cuối dãy có `3` phần tử nên có `2` cặp.
3. `for i in range(k): t[i] = (t[i] + t[i + 1]) % 10`: ghi `k` phần tử mới vào `t[0..k-1]`, duyệt **từ trái sang phải** để `t[i + 1]` còn là giá trị cũ.
4. Khi xong, hai chữ số cuối là `t[0]` và `t[1]`.

In [12]:
def has_same_digits_inplace(s: str) -> bool:
    t = list(map(int, s))
    n = len(t)
    for k in range(n - 1, 1, -1):           # k = số phần tử mới của bước này: n-1, n-2, ..., 2
        for i in range(k):
            t[i] = (t[i] + t[i + 1]) % 10   # trái -> phải: t[i+1] còn là giá trị cũ
    return t[0] == t[1]


for s, expected in examples:
    print(has_same_digits_inplace(s), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )


In [13]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionInplace:
    def hasSameDigits(self, s: str) -> bool:
        t = list(map(int, s))
        n = len(t)
        for k in range(n - 1, 1, -1):
            for i in range(k):
                t[i] = (t[i] + t[i + 1]) % 10
        return t[0] == t[1]


print(SolutionInplace().hasSameDigits("3902"))      # True

True


**Độ phức tạp Cách 2:** **Time `O(n²)`** (`(n-1) + (n-2) + ... + 2` phép cộng), **Space `O(n)`** (list số ban đầu; không tạo thêm list mỗi bước).

### Cách 3 — `zip` + list comprehension

**Giải thích.** Mỗi thao tác viết thành **một dòng**: `t = [(a + b) % 10 for a, b in zip(t, t[1:])]`. `zip(t, t[1:])` cho các cặp liền kề, list comprehension tính chữ số mới. Lặp `while len(t) > 2`. Dễ đọc, nhưng mỗi bước tạo list mới (`O(n)` bộ nhớ thêm, tạm thời).

In [14]:
def has_same_digits_zip(s: str) -> bool:
    t = list(map(int, s))
    while len(t) > 2:
        t = [(a + b) % 10 for a, b in zip(t, t[1:])]      # các cặp liền kề -> chữ số mới
    return t[0] == t[1]


for s, expected in examples:
    print(has_same_digits_zip(s), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )


**Độ phức tạp Cách 3:** **Time `O(n²)`**, **Space `O(n)`**.

### Cách 4 — Công thức tổ hợp với `math.comb`

**Giải thích từng đoạn.**

1. `m = n - 2` là số thao tác; hàng Pascal thứ `m`: `row[i] = comb(m, i)` với `i = 0..m`.
2. `a = Σ row[i] · t[i]`, `b = Σ row[i] · t[i + 1]` (cả hai `mod 10`).
3. Trả `a == b`. Không cần mô phỏng từng bước: chỉ `O(n)` số hạng. (`comb(98, 49)` là số khoảng `10^28` — Python xử lý số nguyên lớn không vấn đề.)

**Vì sao công thức đúng** (quy nạp): sau `0` thao tác, phần tử `j` là `t[j] = C(0,0)·t[j]`. Giả sử sau `m` thao tác phần tử `j` bằng `Σ C(m, i) t[j+i]`. Thao tác kế tiếp cộng hai phần tử liền kề `j` và `j + 1`: `Σ C(m, i) t[j+i] + Σ C(m, i) t[j+1+i]`; gộp hệ số của `t[j+i]` là `C(m, i) + C(m, i-1) = C(m+1, i)` (đẳng thức Pascal). ∎

In [15]:
def has_same_digits_comb(s: str) -> bool:
    t = list(map(int, s))
    m = len(t) - 2                                          # số thao tác
    row = [comb(m, i) for i in range(m + 1)]                # hàng Pascal thứ m
    a = sum(row[i] * t[i] for i in range(m + 1)) % 10
    b = sum(row[i] * t[i + 1] for i in range(m + 1)) % 10
    return a == b


for s, expected in examples:
    print(has_same_digits_comb(s), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )


**Độ phức tạp Cách 4:** tính `n - 1` hệ số và hai tổng ⇒ **Time `O(n)` phép tính** (mỗi `comb` là số lớn nên cụ thể tốn nhiều hơn một chút, vẫn rất nhỏ với `n <= 100`), **Space `O(n)`**.

### Cách 5 — Hàng Pascal tính bằng cộng dần `mod 10`

**Giải thích từng đoạn.** Thay vì `math.comb` (số lớn), ta dựng hàng Pascal **theo đúng quy tắc cộng** và lấy `mod 10` ngay từ đầu: `new_row[i] = (row[i-1] + row[i]) % 10` (đầu và cuối luôn `1`). Vì `mod 10` tương thích với phép cộng, các hệ số luôn nhỏ `0..9`. Kết quả cuối giống Cách 4.

Cách này **không cần `math.comb`** và cho thấy hệ số Pascal cũng chính là kết quả của... một phép mô phỏng cộng-lấy-dư (nhưng chỉ trên hàng hệ số thay vì trên chuỗi số).

In [16]:
def pascal_row_mod10(m: int) -> list:
    row = [1]
    for _ in range(m):
        row = [1] + [(row[i] + row[i + 1]) % 10 for i in range(len(row) - 1)] + [1]
    return row


def has_same_digits_pascal(s: str) -> bool:
    t = list(map(int, s))
    m = len(t) - 2
    row = pascal_row_mod10(m)                               # m + 1 hệ số, đã mod 10
    a = sum(row[i] * t[i] for i in range(m + 1)) % 10
    b = sum(row[i] * t[i + 1] for i in range(m + 1)) % 10
    return a == b


print(pascal_row_mod10(3), pascal_row_mod10(5))            # [1, 3, 3, 1] [1, 5, 0, 0, 5, 1]
for s, expected in examples:
    print(has_same_digits_pascal(s), "(kỳ vọng", expected, ")")

[1, 3, 3, 1] [1, 5, 0, 0, 5, 1]
True (kỳ vọng True )
False (kỳ vọng False )


**Độ phức tạp Cách 5:** dựng hàng Pascal bằng cộng dần tốn `O(m²)` (hàng thứ `j` có `j + 1` phần tử), tổng cộng **Time `O(n²)`** nhưng với phép cộng số nhỏ; **Space `O(n)`**. (Nếu dùng được công thức trực tiếp `C(m, i)` hoặc định lý Lucas thì nhanh hơn — đó là ý của bài II.)

## 8. TRACE từng bước

Ta trace Cách 2 (tại chỗ) trên `s = "34789"` bằng cách in `t` sau mỗi bước, và trace Cách 4 (tổ hợp) cũng trên `s = "34789"`.

In [17]:
def trace_inplace(s):
    t = list(map(int, s))
    n = len(t)
    print(f"s = {s!r}, n = {n}, cần {n - 2} thao tác; t = {t}")
    step = 0
    for k in range(n - 1, 1, -1):
        step += 1
        print(f"  thao tác {step}: tính {k} phần tử mới")
        for i in range(k):
            old_i, old_next = t[i], t[i + 1]
            t[i] = (old_i + old_next) % 10
            print(f"    i={i}: ({old_i} + {old_next}) % 10 = {t[i]}")
        print(f"    t có nghĩa = {t[:k]}")
    print(f"  hai chữ số cuối: {t[0]}, {t[1]} -> {t[0] == t[1]}")
    return t[0] == t[1]


assert trace_inplace("3902") is True
assert trace_inplace("34789") is False

s = '3902', n = 4, cần 2 thao tác; t = [3, 9, 0, 2]
  thao tác 1: tính 3 phần tử mới
    i=0: (3 + 9) % 10 = 2
    i=1: (9 + 0) % 10 = 9
    i=2: (0 + 2) % 10 = 2
    t có nghĩa = [2, 9, 2]
  thao tác 2: tính 2 phần tử mới
    i=0: (2 + 9) % 10 = 1
    i=1: (9 + 2) % 10 = 1
    t có nghĩa = [1, 1]
  hai chữ số cuối: 1, 1 -> True
s = '34789', n = 5, cần 3 thao tác; t = [3, 4, 7, 8, 9]
  thao tác 1: tính 4 phần tử mới
    i=0: (3 + 4) % 10 = 7
    i=1: (4 + 7) % 10 = 1
    i=2: (7 + 8) % 10 = 5
    i=3: (8 + 9) % 10 = 7
    t có nghĩa = [7, 1, 5, 7]
  thao tác 2: tính 3 phần tử mới
    i=0: (7 + 1) % 10 = 8
    i=1: (1 + 5) % 10 = 6
    i=2: (5 + 7) % 10 = 2
    t có nghĩa = [8, 6, 2]
  thao tác 3: tính 2 phần tử mới
    i=0: (8 + 6) % 10 = 4
    i=1: (6 + 2) % 10 = 8
    t có nghĩa = [4, 8]
  hai chữ số cuối: 4, 8 -> False


In [18]:
def trace_comb(s):
    t = list(map(int, s))
    m = len(t) - 2
    row = [comb(m, i) for i in range(m + 1)]
    print(f"s = {s!r}, m = {m}, hàng Pascal = {row}")
    terms_a = [row[i] * t[i] for i in range(m + 1)]
    terms_b = [row[i] * t[i + 1] for i in range(m + 1)]
    a, b = sum(terms_a) % 10, sum(terms_b) % 10
    print(f"  a: các số hạng {terms_a} -> tổng {sum(terms_a)} -> mod 10 = {a}")
    print(f"  b: các số hạng {terms_b} -> tổng {sum(terms_b)} -> mod 10 = {b}")
    return a == b


assert trace_comb("34789") is False
assert trace_comb("3902") is True

s = '34789', m = 3, hàng Pascal = [1, 3, 3, 1]
  a: các số hạng [3, 12, 21, 8] -> tổng 44 -> mod 10 = 4
  b: các số hạng [4, 21, 24, 9] -> tổng 58 -> mod 10 = 8
s = '3902', m = 2, hàng Pascal = [1, 2, 1]
  a: các số hạng [3, 18, 0] -> tổng 21 -> mod 10 = 1
  b: các số hạng [9, 0, 2] -> tổng 11 -> mod 10 = 1


Bảng trace Cách 2 với `s = "34789"` (`t` ban đầu `[3, 4, 7, 8, 9]`):

| thao tác | `k` | các phép tính | `t[:k]` sau bước |
|---|---|---|---|
| 1 | 4 | `3+4→7`, `4+7→1`, `7+8→5`, `8+9→7` | `[7, 1, 5, 7]` |
| 2 | 3 | `7+1→8`, `1+5→6`, `5+7→2` | `[8, 6, 2]` |
| 3 | 2 | `8+6→4`, `6+2→8` | `[4, 8]` |

Kết quả `t[0] = 4`, `t[1] = 8` ⇒ khác nhau ⇒ `false`. Bảng trace Cách 4: `m = 3`, hàng `[1, 3, 3, 1]`; `a = 3 + 12 + 21 + 8 = 44 → 4`; `b = 4 + 21 + 24 + 9 = 58 → 8`; khác nhau ⇒ `false`.

## 9. Code nộp LeetCode

Mình nộp Cách 2 (mô phỏng tại chỗ): `O(n²)` với `n <= 100`, ngắn, dễ kiểm tra và dễ giải thích.

In [19]:
class Solution:
    def hasSameDigits(self, s: str) -> bool:
        t = list(map(int, s))
        n = len(t)
        for k in range(n - 1, 1, -1):               # n - 2 thao tác, bước nào cũng tính k phần tử mới
            for i in range(k):
                t[i] = (t[i] + t[i + 1]) % 10       # duyệt trái -> phải
        return t[0] == t[1]


sol = Solution()
for s, expected in examples:
    print(repr(s), "->", sol.hasSameDigits(s), "(kỳ vọng", expected, ")")

'3902' -> True (kỳ vọng True )
'34789' -> False (kỳ vọng False )


## 10. Vì sao thuật toán đúng

- **Đúng số thao tác:** mỗi thao tác làm độ dài giảm `1`; từ `n` về `2` cần đúng `n - 2` thao tác. Vòng `for k in range(n - 1, 1, -1)` có đúng `(n - 1) - 2 + 1 = n - 2` giá trị (`n - 1, ..., 2`).
- **Mỗi bước đúng định nghĩa:** ở bước với `k` phần tử mới, `t[0..k]` đang giữ dãy hiện tại (độ dài `k + 1`); phần tử mới `i` là `(t[i] + t[i + 1]) % 10` với `i = 0..k-1`. Duyệt trái → phải nên khi tính `t[i]`, `t[i + 1]` vẫn là giá trị **cũ** (chưa bị ghi đè) — chỉ `t[0..i-1]` đã bị ghi.
- **Phần tử thừa không ảnh hưởng:** sau bước, `t[k]` (phần tử cuối của dãy cũ) không còn dùng nữa vì các bước sau chỉ đọc `t[0..k-1]` (ở bước kế tiếp `k' = k - 1` đọc tới `t[k']` = `t[k - 1]`).
- **Kết luận:** sau `n - 2` bước, `t[0], t[1]` là hai chữ số cuối cùng đúng theo đề; so sánh `t[0] == t[1]` là câu trả lời. Công thức Pascal (Cách 4, 5) được chứng minh bằng quy nạp ở mục 7 và được đối chiếu bằng test ngẫu nhiên ở mục 12.
- **Chữ số `0` đứng đầu:** ta làm việc trên list số nên không mất `0` đầu (khác với chuyển cả chuỗi sang `int`).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)`.

| Cách | Time | Space |
|---|---|---|
| 1. Chuỗi mới sau mỗi bước | `O(n²)` | `O(n)` |
| 2. Tại chỗ trên list số | `O(n²)` | `O(n)` |
| 3. `zip` + list comprehension | `O(n²)` | `O(n)` |
| 4. Công thức `math.comb` | `O(n)` số hạng (số lớn) | `O(n)` |
| 5. Hàng Pascal cộng dần `mod 10` | `O(n²)` | `O(n)` |

Với `n <= 100` mọi cách đều tức thì. Công thức tổ hợp (Cách 4) là bước đệm để giải **bài II** (`n` tới `10^5`), nơi mô phỏng `O(n²)` không còn kịp — khi đó người ta tính `C(m, i) mod 10` bằng định lý Lucas (mod 2 và mod 5) rồi ghép lại.

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng bằng `assert`.

In [20]:
sol = Solution()
impls = {
    "string": has_same_digits_string,
    "inplace": has_same_digits_inplace,
    "zip": has_same_digits_zip,
    "comb": has_same_digits_comb,
    "pascal": has_same_digits_pascal,
    "leetcode": sol.hasSameDigits,
}


def check_all(s, expected, label):
    for key, fn in impls.items():
        actual = fn(s)
        assert actual == expected, (label, key, s, actual, expected)
    print("PASS", label)


for idx, (s, expected) in enumerate(examples, 1):
    check_all(s, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2


### 12.1. Các trường hợp biên

Các chuỗi dưới đây là **tự dựng**; đáp án tính tay (ghi trong nhãn) và được máy kiểm tra lại.

In [21]:
check_all("000", True, "độ dài nhỏ nhất 3, toàn 0: 0,0 -> '00'")
check_all("123", False, "một thao tác: 1+2=3, 2+3=5 -> '35'")
check_all("112", False, "1+1=2, 1+2=3 -> '23'")
check_all("505", True, "5+0=5, 0+5=5 -> '55'")
check_all("100", False, "1+0=1, 0+0=0 -> '10'")
check_all("999", True, "9+9=18->8, 9+9->8 -> '88'")
check_all("1111", True, "n = 4: '1111' -> '222' -> '44'")
check_all("0123", False, "n = 4: '0123' -> '135' -> '48' (0 đứng đầu vẫn tính)")
check_all("0" * 100, True, "n = 100, toàn 0")
check_all("9" * 100, True, "n = 100, toàn 9: mọi bước đều ra cùng một chữ số")
check_all("5" + "0" * 99, False, "n = 100: chữ số thứ nhất = 5 (chỉ số hạng s[0] khác 0), chữ số thứ hai = 0 -> khác nhau")

PASS độ dài nhỏ nhất 3, toàn 0: 0,0 -> '00'
PASS một thao tác: 1+2=3, 2+3=5 -> '35'
PASS 1+1=2, 1+2=3 -> '23'
PASS 5+0=5, 0+5=5 -> '55'
PASS 1+0=1, 0+0=0 -> '10'
PASS 9+9=18->8, 9+9->8 -> '88'
PASS n = 4: '1111' -> '222' -> '44'
PASS n = 4: '0123' -> '135' -> '48' (0 đứng đầu vẫn tính)
PASS n = 100, toàn 0
PASS n = 100, toàn 9: mọi bước đều ra cùng một chữ số
PASS n = 100: chữ số thứ nhất = 5 (chỉ số hạng s[0] khác 0), chữ số thứ hai = 0 -> khác nhau


### 12.2. Phản ví dụ cho các cách làm sai

Hai cách sai điển hình: (a) **thiếu `% 10`**; (b) **cập nhật tại chỗ nhưng duyệt từ phải sang trái**.

In [22]:
def wrong_no_mod(s):
    t = list(map(int, s))
    while len(t) > 2:
        t = [a + b for a, b in zip(t, t[1:])]           # SAI: quên % 10
    return t[0] == t[1]


def wrong_reverse_order(s):
    t = list(map(int, s))
    n = len(t)
    for k in range(n - 1, 1, -1):
        for i in range(k - 1, -1, -1):                  # SAI: duyệt phải -> trái
            t[i] = (t[i] + t[i + 1]) % 10
    return t[0] == t[1]


# tìm vài input để thấy hai hàm sai khác Cách 2
rng = random.Random(1)
found_a = found_b = None
for _ in range(2000):
    s = "".join(rng.choice("0123456789") for _ in range(rng.randint(3, 8)))
    ref = has_same_digits_inplace(s)
    if found_a is None and wrong_no_mod(s) != ref:
        found_a = s
    if found_b is None and wrong_reverse_order(s) != ref:
        found_b = s
print("thiếu % 10 sai ở:", found_a, "| duyệt ngược sai ở:", found_b)
assert found_a is not None and found_b is not None
assert wrong_no_mod(found_a) != has_same_digits_inplace(found_a)
assert wrong_reverse_order(found_b) != has_same_digits_inplace(found_b)

thiếu % 10 sai ở: 195364 | duyệt ngược sai ở: 6859


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh chuỗi chữ số **đúng ràng buộc** (độ dài `3..100`, chỉ gồm `0..9`, được phép có số `0` đứng đầu), kể cả các bảng chữ số nhỏ (`"01"`, `"05"`) để có nhiều ca đáp án `True`. Mọi cách phải khớp với Cách 1 (mô phỏng đúng đề).

In [23]:
def valid(s):
    return 3 <= len(s) <= 100 and all(c in "0123456789" for c in s)


rng = random.Random(3461)
count_true = count_false = 0
for trial in range(3000):
    n = rng.randint(3, 100)
    alphabet = rng.choice(["0123456789", "01", "05", "19", "0246"])
    s = "".join(rng.choice(alphabet) for _ in range(n))
    assert valid(s)
    ref = has_same_digits_string(s)
    for key, fn in impls.items():
        assert fn(s) == ref, (key, s)
    if ref:
        count_true += 1
    else:
        count_false += 1
print("PASS 3000 chuỗi ngẫu nhiên đúng ràng buộc; số ca True =", count_true, ", False =", count_false)
assert count_true > 100 and count_false > 100

# vét cạn mọi chuỗi 4 chữ số (10^4 chuỗi, kể cả số 0 đứng đầu)
for x in range(10000):
    s = f"{x:04d}"
    ref = has_same_digits_string(s)
    assert has_same_digits_inplace(s) == ref and has_same_digits_comb(s) == ref and has_same_digits_pascal(s) == ref
print("PASS vét cạn 10000 chuỗi 4 chữ số")

PASS 3000 chuỗi ngẫu nhiên đúng ràng buộc; số ca True = 692 , False = 2308
PASS vét cạn 10000 chuỗi 4 chữ số


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Cộng ký tự thay vì cộng số

In [24]:
def sai_cong_ky_tu(s):
    t = list(s)                                          # SAI: các phần tử vẫn là ký tự
    while len(t) > 2:
        t = [(a + b) % 10 for a, b in zip(t, t[1:])]
    return t[0] == t[1]


try:
    sai_cong_ky_tu("3902")
except TypeError as e:
    print("TypeError:", e)
print("đúng:", Solution().hasSameDigits("3902"))

TypeError: not all arguments converted during string formatting
đúng: True


**Vì sao sai:** `"3" + "9"` là **nối chuỗi** ra `"39"`, rồi `"39" % 10` là phép định dạng chuỗi và báo `TypeError`. Phải đổi sang số: `int(c)` hoặc `list(map(int, s))`. **Cách sửa:** `t = list(map(int, s))`.

### Lỗi 2: Quên `% 10`

In [25]:
def sai_quen_mod(s):
    t = list(map(int, s))
    while len(t) > 2:
        t = [a + b for a, b in zip(t, t[1:])]            # SAI: không lấy dư
    return t[0] == t[1]


for s in ["3902", "34789", "999"]:
    print(s, "sai:", sai_quen_mod(s), "| đúng:", Solution().hasSameDigits(s))

3902 sai: False | đúng: True
34789 sai: False | đúng: False
999 sai: True | đúng: True


**Vì sao sai:** đề yêu cầu mỗi chữ số mới là **tổng `mod 10`** (luôn trong `0..9`). Không `% 10` thì các số lớn dần và hai số cuối gần như không bao giờ bằng nhau. Ví dụ `"3902"` đáp án đúng là `True` nhưng bản sai cho `False`. **Cách sửa:** `(a + b) % 10`.

### Lỗi 3: Dừng sai — `len > 1` thay vì `len > 2`

In [26]:
def sai_dung_o_mot(s):
    t = list(map(int, s))
    while len(t) > 1:                                     # SAI: tiếp tục tới còn 1 phần tử
        t = [(a + b) % 10 for a, b in zip(t, t[1:])]
    return len(t) == 2 and t[0] == t[1]


print("sai:", sai_dung_o_mot("3902"), "| đúng:", Solution().hasSameDigits("3902"))

sai: False | đúng: True


**Vì sao sai:** đề dừng khi còn **đúng hai** chữ số rồi mới so sánh. Nếu cứ làm tiếp thì còn một chữ số và không còn "hai chữ số cuối" để so. **Cách sửa:** `while len(t) > 2` (hoặc đúng `n - 2` thao tác).

### Lỗi 4: Cập nhật tại chỗ nhưng duyệt từ phải sang trái

In [27]:
def sai_duyet_nguoc(s):
    t = list(map(int, s))
    n = len(t)
    for k in range(n - 1, 1, -1):
        for i in range(k - 1, -1, -1):                    # SAI: duyệt phải -> trái
            t[i] = (t[i] + t[i + 1]) % 10                 # t[i + 1] đã bị ghi đè ở bước trước đó trong cùng vòng
        # (t[k] không còn dùng nữa)
    return t[0] == t[1]


rng = random.Random(7)
bad = None
for _ in range(2000):
    s = "".join(rng.choice("0123456789") for _ in range(rng.randint(3, 8)))
    if sai_duyet_nguoc(s) != Solution().hasSameDigits(s):
        bad = s
        break
print("ví dụ input làm bản sai khác bản đúng:", bad, "| sai:", sai_duyet_nguoc(bad), "| đúng:", Solution().hasSameDigits(bad))

ví dụ input làm bản sai khác bản đúng: 26018 | sai: True | đúng: False


**Vì sao sai:** khi tính `t[i]` ta cần `t[i + 1]` **cũ**; duyệt phải → trái thì `t[i + 1]` đã được ghi bằng giá trị **mới** ở bước trước của cùng vòng lặp. **Cách sửa:** duyệt trái → phải (hoặc tạo list mới bằng `zip`).

### Lỗi 5: Chuyển cả chuỗi sang `int` làm mất số `0` đứng đầu

In [28]:
def sai_mat_so_0(s):
    x = int(s)                                            # SAI: "0123" -> 123, mất số 0 đứng đầu
    s = str(x)
    while len(s) > 2:
        s = "".join(str((int(a) + int(b)) % 10) for a, b in zip(s, s[1:]))
    return len(s) == 2 and s[0] == s[1]


print("int('0123') =", int("0123"))
print("sai:", sai_mat_so_0("0011"), "| đúng:", Solution().hasSameDigits("0011"))
print("sai:", sai_mat_so_0("0000"), "| đúng:", Solution().hasSameDigits("0000"))

int('0123') = 123
sai: True | đúng: False
sai: False | đúng: True


**Vì sao sai:** `s` là **chuỗi** chữ số, mỗi vị trí là một chữ số riêng; `int(s)` coi cả chuỗi là một số nên **bỏ các số `0` đứng đầu** (`"0011"` thành `11`, chỉ còn hai chữ số nên bản sai so luôn `1 == 1` ra `True`, trong khi đáp án đúng là `False`; `"0000"` thành `0`, chỉ còn một chữ số nên bản sai trả `False` thay vì `True`). **Cách sửa:** không dùng `int(s)`; chỉ đổi **từng ký tự** (`list(map(int, s))`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề nói "lặp thao tác trên cặp liền kề cho đến khi còn ... phần tử" ⇒ **mô phỏng** (ràng buộc nhỏ cho phép `O(n²)`).
- Số thao tác = (độ dài đầu) − (độ dài đích). Ở đây `n - 2`.
- Cặp liền kề: `zip(t, t[1:])` (ngắn gọn) hoặc `range(len(t) - 1)` (có chỉ số).
- Chữ số trong chuỗi: luôn `int(c)` trước khi cộng; **không** `int(s)` cả chuỗi nếu có thể có số `0` đứng đầu.
- Khi cập nhật tại chỗ: nghĩ xem phần tử "bên cạnh" còn là giá trị cũ không ⇒ chọn đúng chiều duyệt.
- Nhận ra **tam giác Pascal**: lấy tổng hai phần tử liền kề liên tiếp ⇒ hệ số là `C(m, i)`. Đây là gợi ý cho bản tối ưu (bài II) dùng `C(m, i) mod 10` (định lý Lucas).
- `mod` thân thiện với cộng và nhân: `(a + b) % 10` tương đương cộng rồi lấy dư một lần ở cuối.
- Kiểm tra nhanh: tự làm tay hai ví dụ trong đề (`"3902"` → `"292"` → `"11"`).

## 15. Biến thể của bài

1. **Tổng tam giác (Triangular Sum):** lặp đến khi còn **một** chữ số rồi trả về chữ số đó (**Find Triangular Sum of an Array (2221)**).
2. **Bài II (Hard, LeetCode 3463):** cùng ý tưởng nhưng `n` tới `10^5`; cần công thức `C(m, i) mod 10` (Lucas mod 2 và mod 5).
3. Đổi `mod 10` thành `mod` số khác (ví dụ `mod 7`), hoặc đổi phép cộng thành `XOR` (liên quan tam giác Sierpinski).
4. In ra **toàn bộ tam giác** các dãy trung gian (như đã trace).
5. Bài cùng họ về tam giác Pascal: **Pascal's Triangle (118)**, **Pascal's Triangle II (119)**.

In [29]:
def triangular_sum(digits):
    """Lặp tới còn 1 phần tử (LeetCode 2221), mod 10."""
    t = list(digits)
    while len(t) > 1:
        t = [(a + b) % 10 for a, b in zip(t, t[1:])]
    return t[0]


def triangular_sum_comb(digits):
    m = len(digits) - 1
    return sum(comb(m, i) * d for i, d in enumerate(digits)) % 10


def all_levels(s):
    """Liệt kê mọi dãy trung gian (từ s đến còn 2 chữ số)."""
    t = list(map(int, s))
    levels = ["".join(map(str, t))]
    while len(t) > 2:
        t = [(a + b) % 10 for a, b in zip(t, t[1:])]
        levels.append("".join(map(str, t)))
    return levels


print(triangular_sum([1, 2, 3, 4, 5]), all_levels("34789"))
assert triangular_sum([1, 2, 3, 4, 5]) == 8                 # LeetCode 2221 ví dụ 1: kết quả 8
assert triangular_sum([5]) == 5
assert triangular_sum_comb([1, 2, 3, 4, 5]) == 8
assert all_levels("34789") == ["34789", "7157", "862", "48"]
assert all_levels("3902") == ["3902", "292", "11"]
for x in range(1000):
    d = [int(c) for c in f"{x:03d}"]
    assert triangular_sum(d) == triangular_sum_comb(d)

8 ['34789', '7157', '862', '48']


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Chuỗi mới sau mỗi bước | mô phỏng đúng đề, `str ↔ int` | `O(n²)` | `O(n)` | hiểu đề, đối chiếu |
| 2. Tại chỗ trên list số | `t[i] = (t[i] + t[i+1]) % 10`, trái → phải | `O(n²)` | `O(n)` | **nên nộp** |
| 3. `zip` + list comprehension | một dòng mỗi thao tác | `O(n²)` | `O(n)` | ngắn gọn, dễ đọc |
| 4. Công thức `math.comb` | hệ số Pascal `C(n-2, i)` | `O(n)` số hạng | `O(n)` | hiểu cấu trúc toán học, bước đệm cho bài II |
| 5. Hàng Pascal cộng dần `mod 10` | dựng hàng hệ số bằng cộng, tránh số lớn | `O(n²)` | `O(n)` | khi không muốn dùng `comb` |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `(7 + 5) % 10`, `list(map(int, "407"))`, `[(a + b) % 10 for a, b in zip([1, 2, 3], [2, 3, 4])]`, `Solution().hasSameDigits("505")`.

**Bài 2 (tính tay).** Với `s = "5489"`: viết từng thao tác (các tổng `mod 10` và chuỗi mới) cho đến khi còn hai chữ số, rồi cho biết đáp án.

**Bài 3 (điền TODO).** Hoàn thành `has_same_digits_todo(s)` theo Cách 3 (`zip` + list comprehension).

In [30]:
def has_same_digits_todo(s):
    # TODO 1: t = list(map(int, s))
    # TODO 2: while len(t) > 2: t = [ (a + b) % 10 for a, b in zip(t, t[1:]) ]
    # TODO 3: return t[0] == t[1]
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `has_same_digits_string2(s)` theo Cách 1 nhưng dùng `"".join(...)` với generator thay cho `new += ...` (mỗi thao tác một dòng trên chuỗi, vẫn **không** dùng `int(s)` cả chuỗi).

**Bài 5 (biến thể / mini challenge).** (a) Viết `final_single_digit(s)`: lặp tới còn **một** chữ số và trả chữ số đó (kiểu LeetCode 2221). (b) Viết `pascal_mod_row(m, mod)` trả hàng `m` của tam giác Pascal nhưng lấy `mod` tuỳ ý, rồi dùng nó để kiểm tra `final_single_digit` bằng công thức `Σ C(n-1, i) · d[i] (mod 10)`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `2`, `[4, 0, 7]`, `[3, 5, 7]`, `True`. Bài 3: ba dòng đúng theo ba TODO.
- **Mức 2 (vừa):** Bài 2: `5489` → `9`(`5+4`), `2`(`4+8=12`), `7`(`8+9=17`) ⇒ `927` → `1`(`9+2=11`), `9`(`2+7`) ⇒ `19` ⇒ `1 != 9` ⇒ `False`. Bài 4: `s = "".join(str((int(a) + int(b)) % 10) for a, b in zip(s, s[1:]))`.
- **Mức 3 (sâu):** Bài 5(a): đổi điều kiện vòng lặp thành `while len(t) > 1`. 5(b): hàng Pascal: `row = [1]; lặp m lần: row = [1] + [(row[i] + row[i + 1]) % mod ...] + [1]`; chú ý với `mod = 1` mọi hệ số là `0` (nên xử lý riêng hoặc đừng dùng `mod = 1`).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert (7 + 5) % 10 == 2
assert list(map(int, "407")) == [4, 0, 7]
assert [(a + b) % 10 for a, b in zip([1, 2, 3], [2, 3, 4])] == [3, 5, 7]
assert Solution().hasSameDigits("505") is True

# Bài 2
assert all_levels("5489") == ["5489", "927", "19"]
assert Solution().hasSameDigits("5489") is False


# Bài 3
def has_same_digits_todo_ans(s):
    t = list(map(int, s))
    while len(t) > 2:
        t = [(a + b) % 10 for a, b in zip(t, t[1:])]
    return t[0] == t[1]


for s, expected in examples:
    assert has_same_digits_todo_ans(s) == expected


# Bài 4
def has_same_digits_string2(s):
    while len(s) > 2:
        s = "".join(str((int(a) + int(b)) % 10) for a, b in zip(s, s[1:]))
    return s[0] == s[1]


for s, expected in examples:
    assert has_same_digits_string2(s) == expected
assert has_same_digits_string2("0123") is False


# Bài 5(a)
def final_single_digit(s):
    t = list(map(int, s))
    while len(t) > 1:
        t = [(a + b) % 10 for a, b in zip(t, t[1:])]
    return t[0]


assert final_single_digit("12345") == 8
assert final_single_digit("7") == 7
assert final_single_digit("34789") == (4 + 8) % 10


# Bài 5(b)
def pascal_mod_row(m, mod):
    row = [1 % mod]
    for _ in range(m):
        row = [1 % mod] + [(row[i] + row[i + 1]) % mod for i in range(len(row) - 1)] + [1 % mod]
    return row


assert pascal_mod_row(3, 10) == [1, 3, 3, 1]
assert pascal_mod_row(5, 10) == [1, 5, 0, 0, 5, 1]
assert pascal_mod_row(4, 2) == [1, 0, 0, 0, 1]
rng = random.Random(19)
for _ in range(300):
    s = "".join(rng.choice("0123456789") for _ in range(rng.randint(1, 30)))
    d = list(map(int, s))
    row = pascal_mod_row(len(d) - 1, 10)
    assert sum(r * x for r, x in zip(row, d)) % 10 == final_single_digit(s)
print("Tất cả đáp án bài tập đều đúng")

Tất cả đáp án bài tập đều đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (mô phỏng tại chỗ trên list số; `n - 2` thao tác)

```python
class Solution:
    def hasSameDigits(self, s: str) -> bool:
        t = list(map(int, s))
        n = len(t)
        for k in range(n - 1, 1, -1):
            for i in range(k):
                t[i] = (t[i] + t[i + 1]) % 10
        return t[0] == t[1]
```

**Checklist**

- [ ] Đã đổi ký tự thành **số** (`int(c)`) trước khi cộng?
- [ ] Có `% 10` cho mỗi tổng?
- [ ] Dừng khi còn **đúng hai** chữ số (`n - 2` thao tác), rồi mới so `t[0] == t[1]`?
- [ ] Nếu cập nhật tại chỗ: duyệt **trái → phải** để `t[i + 1]` còn là giá trị cũ?
- [ ] Không dùng `int(s)` cho cả chuỗi (giữ số `0` đứng đầu)?
- [ ] Đã thử `"3902"` (`True`), `"34789"` (`False`), `"000"` (`True`), `"0123"` (`False`)?
- [ ] Skeleton đúng `class Solution` / `def hasSameDigits(self, s: str) -> bool`?